In [ ]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import pandas as pd
import random
import re

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from huggingface_hub import login
from google.colab import userdata

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

HF_TOKEN = userdata.get('HF')
login(token=HF_TOKEN)

!pip install -q -U bitsandbytes

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/LR2.csv')
df_dev = pd.read_csv('/content/drive/MyDrive/LR2_dev.csv')
df_answers = pd.read_csv('/content/drive/MyDrive/LR2_dev_answer.csv')

In [ ]:
FILE_TO_BOOK_TITLE = {
    "Bulichev_Gorod_bez_pamyati": "Город без памяти",
    "Bulichev_Lilovii_shar": "Лиловый шар",
    "Bulichev_Sto_let_tomu_vpered": "Сто лет тому вперёд",

    "Bulgakov_KrescheniePovorotom": "Крещение поворотом",
    "Bulgakov_MasterIMargarita.txt": "Мастер и Маргарита",
    "Bulgakov_Morfiyi": "Морфий",
    "Bulgakov_PolotenceSPetuhom": "Полотенце с петухом",
    "Bulgakov_PriklyucheniyaPokoynika": "Приключения покойника",
    "Bulgakov_RokovyeYayca": "Роковые яйца",
    "Bulgakov_StalnoeGorlo": "Стальное горло",
    "Bulgakov_TmaEgipetskaya": "Тьма египетская",
    "Bulgakov_ZvezdnayaSyp": "Звездная сыпь",

    "Gogol_KakPossorilsyaIvanIvanovichSIvanomNikiforovichem": "Как поссорился Иван Иванович с Иваном Никифоровичем",
    "Gogol_MertvieDushi": "Мертвые души",
    "Gogol_StarosvetskiyePomeshchiki": "Старосветские помещики",
    "Gogol_TarasBulba": "Тарас Бульба",
    "Gogol_Vii": "Вий",
    "Gogol_Sbornik": None,

    "Gorkiyi_DeloArtamonovyih": "Дело Артамоновых",

    "Lermontov_KnyaginyaLigovskaya": "Княгиня Лиговская",

    "Tolstoy_Decabristy": "Декабристы",
    "Tolstoy_VoynaIMir1": "Война и мир",
    "Tolstoy_VoynaIMir2": "Война и мир",

    "Turgenev_Asya": "Ася",
    "Turgenev_Mumu": "Муму",
    "Turgenev_ZapiskiOhotnika": "Записки охотника"
}

In [ ]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu
!pip install pymorphy2
!pip install transformers accelerate torch sentencepiece

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

def load_literary_corpus(corpus_root):
  documents = []
  allowed_suffixes = {'.txt', '.TXT'}
  txt_files = [p for p in Path(corpus_root).rglob('*') if p.suffix in allowed_suffixes]

  for file_path in txt_files:
    author = file_path.parent.name
    book_title = file_path.stem
    if book_title not in FILE_TO_BOOK_TITLE:
      print(book_title)
      print('ЧЕ ТА НЕ ТО БАМ БАМ')
      continue
    book_title = FILE_TO_BOOK_TITLE[book_title]
    if book_title is None:
      continue
    print(book_title)
    loader = TextLoader(str(file_path), encoding='utf-8')
    docs = loader.load()
    for doc in docs:
        doc.metadata.update({
            'source': str(file_path),
            'author': author,
            'book_title': book_title,
            'file_name': file_path.name,
            'encoding': 'utf-8'
        })
        documents.append(doc)
  return documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_literary_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=300,
        separators=[
            "\n\n",
            "\n",
            "— ", "– ",
            ". ", "! ", "? ",
            "; ", ", ",
            " ",
            ""
        ],
        length_function=len,
        is_separator_regex=False,
    )

    chunks = text_splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = i

    print(f"Создано чанков: {len(chunks)}")
    print(f"Средний размер: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} символов")

    return chunks

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
import numpy as np

In [ ]:
!pip install -q sentence-transformers chromadb torch

In [ ]:
class E5Embeddings:
    def __init__(self, model_name="intfloat/multilingual-e5-large"):
        self.model = SentenceTransformer(model_name)
        self.model.max_seq_length = 512

    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        embeddings = self.model.encode(texts, normalize_embeddings=True, batch_size=32)
        return embeddings.tolist()

    def embed_query(self, text):
        text = f"query: {text}"
        embeddings = self.model.encode([text], normalize_embeddings=True)
        return embeddings[0].tolist()

In [ ]:
def create_vector_store(chunks, persist_path="/content/drive/MyDrive/chroma_index"):
    embeddings = E5Embeddings("intfloat/multilingual-e5-large")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_path
    )
    vector_store.persist()

    print(f"Обработано чанков: {len(chunks)}")
    print(f"Размерность эмбеддингов: {embeddings.model.get_sentence_embedding_dimension()}")

    return vector_store, embeddings

In [ ]:
!pip install -q nltk

!pip install rank_bm25

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
import re
import pymorphy2

from nltk.stem.snowball import SnowballStemmer

In [ ]:
stemmer = SnowballStemmer("russian")

In [ ]:
class HybridRetriever:
    def __init__(self, vector_store, chunks):
        self.vector_store = vector_store
        self.chunks = chunks
        # self.morph = pymorphy2.MorphAnalyzer()

    # def _lemmatize(self, text):
    #     words = re.findall(r'[а-яёa-z]+', text.lower())
    #     lemmas = []
    #     for w in words:
    #         if len(w) > 2:
    #             lemmas.append(self.morph.parse(w)[0].normal_form)
    #     return lemmas

    # def _tokenize(self, text):
    #     text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text.lower())
    #     return [w for w in text.split() if len(w) > 2]

    def _stem(self, text):
        words = nltk.word_tokenize(text.lower())
        return [stemmer.stem(w) for w in words if len(w) > 2 and w.isalpha()]

    def retrieve(self, query, book_title, top_k=5, use_bm25=True):
        dense_results = self.vector_store.similarity_search_with_relevance_scores(
            query,
            k=top_k * 2 if use_bm25 else top_k,
            filter={"book_title": book_title}
        )

        dense_chunks = [doc for doc, _ in dense_results]
        dense_scores = [score for _, score in dense_results]

        if not dense_chunks:
            print(f"Нет чанков для книги '{book_title}'")
            return []

        if not use_bm25:
            return dense_chunks[:top_k]

        tokenized_corpus = [self._stem(c.page_content) for c in dense_chunks]
        bm25 = BM25Okapi(tokenized_corpus)
        bm25_scores = bm25.get_scores(self._stem(query))
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        rrf_k = 80
        rrf_scores = {}

        for i in range(len(dense_chunks)):
            dense_rank = i
            sparse_rank = np.where(bm25_ranks == i)[0][0]

            rrf_scores[i] = 1 / (rrf_k + dense_rank + 1) + 1 / (rrf_k + sparse_rank + 1)

        top_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
        return [dense_chunks[i] for i in top_indices]

In [ ]:
corpus = load_literary_corpus("/content/drive/MyDrive/dataset/dataset")

In [ ]:
chunks = chunk_literary_documents(corpus)

In [ ]:
vector_store, embeddings = create_vector_store(chunks)

In [ ]:
retriever = HybridRetriever(vector_store, chunks)

In [ ]:
def generate_answer(question, options, book_title, context_chunks):
    context = "\n\n".join([chunk.page_content for chunk in context_chunks[:5]])
    prompt = f"""Ты — эксперт по русской литературе. Ответь ТОЛЬКО цифрой (1, 2, 3 или 4) на основе контекста.

Вопрос: {question}

Варианты ответов:
1. {options[0]}
2. {options[1]}
3. {options[2]}
4. {options[3]}

Контекст из произведения "{book_title}":
{context}

Ответ (ТОЛЬКО цифра 1/2/3/4):"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )


    answer_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    match = re.search(r'[оО]твет[:\s]*([1-4])', answer_text)
    if not match:
        match = re.search(r'([1-4])[^\d]*$', answer_text)
    if not match:
        match = re.search(r'[1-4]', answer_text)
    return match.group(1) if match else "1"

In [ ]:
BOOK_NAME_CORRECTIONS = {
    "Старосветских помещиках": "Старосветские помещики",
    "Крещении поворотом": "Крещение поворотом",
    "Полотенце с Петухом": "Полотенце с петухом"
}

In [ ]:
res = {'answer': []}
for idx, row in df.iterrows():
  question = row['question']
  options = [row['answer a'], row['answer b'], row['answer c'], row['answer d']]
  book_from_dataset = row['book']
  if book_from_dataset in BOOK_NAME_CORRECTIONS:
    book_from_dataset = BOOK_NAME_CORRECTIONS[book_from_dataset]

  relevant_chunks = retriever.retrieve(question, book_from_dataset, top_k=10)

  answer = generate_answer(question, options, book_from_dataset, relevant_chunks)
  res['answer'].append(answer)
df_an = pd.DataFrame(res)
df_an.to_csv('answers.csv')